# ecoli-0_vs_1: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('10_ecoli-0_vs_1_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,96.902,96.586,98.824,94.667,99.286,96.976,99.278,99.448,96.744
smote,96.902,96.586,98.824,94.667,99.286,96.976,99.068,99.337,96.744
bsmote,96.902,96.586,98.824,94.667,99.286,96.976,97.901,98.935,96.744
ros,96.902,96.586,98.824,94.667,99.286,96.976,99.278,99.448,96.744
adasyn,96.890,96.306,97.647,95.333,98.596,96.965,98.538,99.090,96.598
casb,96.902,96.586,98.824,94.667,99.286,96.976,99.241,99.426,96.744
casb_relaxed,96.902,96.586,98.824,94.667,99.286,96.976,99.278,99.448,96.744
casb_edited,96.719,96.301,98.301,94.667,98.929,96.798,99.119,99.358,96.510
casb_ros,96.902,96.586,98.824,94.667,99.286,96.976,99.278,99.448,96.744


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.194,95.014,100.000,90.750,100.000,95.375,98.065,98.414,95.104
smote,95.217,94.787,98.708,91.417,99.298,95.357,98.031,98.391,95.002
bsmote,95.906,95.252,97.495,93.375,98.608,95.992,97.935,98.323,95.579
ros,95.020,94.712,99.375,90.750,99.643,95.196,98.028,98.391,94.866
adasyn,95.347,94.141,94.837,94.000,96.872,95.436,96.977,98.139,94.744
casb,95.582,95.184,98.708,92.083,99.298,95.691,98.031,98.391,95.383
casb_relaxed,96.239,95.895,98.787,93.375,99.286,96.330,98.072,98.414,96.067
casb_edited,96.199,95.610,97.634,94.000,98.584,96.292,98.075,98.403,95.904
casb_ros,95.426,94.865,97.946,92.083,98.953,95.518,98.028,98.391,95.146


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
smote,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
bsmote,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
ros,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
adasyn,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
casb,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
casb_relaxed,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
casb_edited,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632
casb_ros,98.644,98.621,100.0,97.333,100.0,98.667,98.752,98.759,98.632


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,97.580,97.276,98.824,96.000,99.286,97.643,99.155,99.310,97.428
smote,97.764,97.323,98.157,96.667,98.941,97.804,99.200,99.356,97.544
bsmote,98.100,97.729,98.301,97.333,98.929,98.131,99.149,99.381,97.915
ros,97.932,97.645,98.824,96.667,99.286,97.976,99.170,99.310,97.789
adasyn,96.884,95.789,95.249,96.667,97.204,96.936,99.082,99.289,96.337
casb,97.764,97.323,98.157,96.667,98.941,97.804,99.254,99.425,97.544
casb_relaxed,97.764,97.323,98.157,96.667,98.941,97.804,99.141,99.335,97.544
casb_edited,97.235,96.414,96.426,96.667,97.894,97.280,99.099,99.312,96.825
casb_ros,97.764,97.323,98.157,96.667,98.941,97.804,99.254,99.425,97.544


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,97.080,96.874,99.412,94.688,99.643,97.165,98.812,98.983,96.977
smote,97.132,96.829,98.922,95.021,99.381,97.201,98.762,98.961,96.980
bsmote,97.388,97.047,98.655,95.677,99.206,97.441,98.434,98.849,97.218
ros,97.124,96.891,99.256,94.854,99.554,97.204,98.807,98.977,97.008
adasyn,96.941,96.214,96.933,95.833,98.168,97.001,98.337,98.819,96.578
casb,97.223,96.928,98.922,95.188,99.381,97.284,98.819,99.000,97.076
casb_relaxed,97.387,97.106,98.942,95.510,99.378,97.444,98.811,98.989,97.247
casb_edited,97.199,96.736,98.090,95.667,98.852,97.259,98.761,98.958,96.968
casb_ros,97.184,96.849,98.732,95.188,99.295,97.241,98.828,99.006,97.016


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,96.902,smote,96.902,0.000,TIE
1,f1,casb,96.586,smote,96.586,0.000,TIE
2,precision,casb,98.824,smote,98.824,0.000,TIE
3,recall,casb,94.667,adasyn,95.333,-0.667,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,96.239,bsmote,95.906,0.333,WIN
1,f1,casb_relaxed,95.895,bsmote,95.252,0.642,WIN
2,precision,casb_relaxed,98.787,ros,99.375,-0.588,LOSS
3,recall,casb_edited,94.000,adasyn,94.000,0.000,TIE


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,98.644,smote,98.644,0.0,TIE
1,f1,casb,98.621,smote,98.621,0.0,TIE
2,precision,casb,100.000,smote,100.000,0.0,TIE
3,recall,casb,97.333,smote,97.333,0.0,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,97.387,bsmote,97.388,-0.001,LOSS
1,f1,casb_relaxed,97.106,bsmote,97.047,0.059,WIN
2,precision,casb_relaxed,98.942,ros,99.256,-0.314,LOSS
3,recall,casb_edited,95.667,adasyn,95.833,-0.167,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
1,42,1,svm,smote,0.75,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
2,42,1,svm,bsmote,0.75,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
3,42,1,svm,ros,0.75,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
4,42,1,svm,adasyn,0.75,0.964901,0.937500,0.882353,1.000000,0.931034,0.965517,0.992157,0.995402,0.951201,None
5,42,1,svm,casb,1.00,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
6,42,1,svm,casb_relaxed,1.00,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
7,42,1,svm,casb_edited,1.00,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,0.995833,0.997701,0.965805,None
8,42,1,svm,casb_ros,0.50,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
9,42,1,knn,original,0.50,0.966092,0.965517,1.000000,0.933333,1.000000,0.966667,1.000000,1.000000,0.965805,None
